# Import a control: ZIP or folder from SharePoint

**What it does**
Turns what ING keeps in SharePoint into a KCT case in one step:

```
<control folder>/Input/Evidence/...
<control folder>/Input/Test Plan and Control Details/...
```

- **ZIP** (SharePoint: select the control folder > Download): the ZIP is kept, unpacked and sorted into `storage/imports/<case>/`:
  - `source/`: the original ZIP, with date and time in the name
  - `Input/Evidence/` and `Input/Test Plan and Control Details/`: the files, sorted
  - `other/`: files that are neither
  - `import.json`: every decision and its reason

  The ZIP is then imported like a folder.
- **Folder** (OneDrive sync, or an unpacked download): imported directly. The source folder is never changed.

The case is named after the control folder (e.g. `5.0 Password resets ...`). Importing the same control again updates that case.

**Who decides what each file is**
- **ZIP with the SharePoint folders**: the folders decide what is evidence. The AI only tells the control details from the test plan in "Test Plan and Control Details".
- **Flat ZIP** (all files mixed): the AI sorts every file into control details, test plan, evidence or other, from its path, name and first pages.
- A wrong decision can be fixed with `OVERRIDES`.

**How to use**
1. Choose the input in Step 1: a fake ZIP made here, your own ZIP (`ZIP_PATH`), or a folder (`FOLDER`).
2. Run all cells. Step 2 prints every decision with its reason.
3. `INTERPRET` / `ASSESS` in Step 2 go on to test steps and the assessment.

With `KCT_AI_PROVIDER=fake` the sorting answers are fixed (control details first, then test plan), so only the structured fake ZIP works there; use Gemini for flat ZIPs.

## Setup

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.app import boot
from services.config import settings

kct = boot()
config = settings()
print("AI provider:", config.ai_provider, "| database:", config.db_driver, "| imports go to", config.storage_root / "imports")

## Step 1: What to import

Set exactly one of them:
- `MAKE_FAKE_ZIP = True`: a fake control in the SharePoint structure, as a ZIP (`ControlFolderFactory`). `FLAT = True` makes it without folders.
- `ZIP_PATH`: your own ZIP, for example downloaded from SharePoint.
- `FOLDER`: a control folder or its `Input` folder, for example the OneDrive-synced one.

In [ ]:
from factories import ControlFolderFactory

MAKE_FAKE_ZIP = True
FLAT = False
ZIP_PATH = ""          # e.g. "/home/jupyter/Downloads/OneDrive_1_2026-11-07.zip"
FOLDER = ""            # e.g. "/home/jupyter/sharepoint/5.0 Password resets"

if MAKE_FAKE_ZIP:
    ZIP_PATH = str(config.storage_root / "demo" / ("control_flat.zip" if FLAT else "control.zip"))
    Path(ZIP_PATH).parent.mkdir(parents=True, exist_ok=True)
    Path(ZIP_PATH).write_bytes(ControlFolderFactory().zip(flat=FLAT))
    print("Made", ZIP_PATH)
print("Importing:", ZIP_PATH or FOLDER)

## Step 2: Import

- `CASE_NAME`: empty means it is taken from the ZIP or folder.
- `OVERRIDES`: fixes a sorting decision, e.g. `{"Test Plan TP-PRC-IAM-014 2026Q1 (SAMPLE).pdf": "test_plan"}`. Roles: `control_details`, `test_plan`, `evidence`, `other`.
- `INTERPRET` / `ASSESS`: continue to the test steps and the assessment.

In [ ]:
CASE_NAME = ""
OVERRIDES = {}
INTERPRET = False
ASSESS = False

if ZIP_PATH:
    result = kct.importer.import_zip(ZIP_PATH, case_name=CASE_NAME, interpret=INTERPRET, assess=ASSESS, progress=print)
else:
    result = kct.importer.import_folder(FOLDER, overrides=OVERRIDES, interpret=INTERPRET, assess=ASSESS, progress=print)
print("\n✅ case", result.case_id)

## Step 3: What went where

The sorting decisions, the evidence copied into the case, and what was skipped.

In [ ]:
print("Control details:", [c.name for c in result.control_details])
print("Test plan:      ", [c.name for c in result.test_plan])
if result.sorted_by_ai:
    print("\nSorted by the AI (flat ZIP):")
    for c in result.sorted_by_ai:
        print(f"   {c.role:<16} {c.name}  ({c.reason})")
print(f"\nEvidence in the case ({len(result.evidence)}):")
for path in result.evidence:
    print("  ", path)
for s in result.skipped:
    print("   skipped:", s.path, "-", s.reason)
if result.removed_evidence:
    print("Removed (no longer in the source):", result.removed_evidence)

## Step 4: The sorted folder and the kept ZIP (ZIP imports)

In [ ]:
if result.sorted_folder:
    root = Path(result.sorted_folder)
    for path in sorted(root.rglob("*")):
        if path.is_file():
            print("  ", path.relative_to(root))
    print("\nSource ZIP:", result.source_zip)

## Next

- Run `11_assessment.ipynb`: it takes the newest case with test steps. Or set `INTERPRET = True` / `ASSESS = True` in Step 2 and run it again.
- The same through the API:

```bash
curl -F file=@OneDrive_1_2026-11-07.zip -F interpret=true http://localhost:8000/api/v1/cases/import-zip
curl -X POST -H 'Content-Type: application/json' \
     -d '{"path": "/home/jupyter/sharepoint/5.0 Password resets", "assess": true}' \
     http://localhost:8000/api/v1/cases/import          # needs KCT_IMPORT_ROOT
```